# LiDAR Spoofing and Defense Practice

The assignment for this notebook is to use geometric operations usign Open3D to inject spoofed points into a LiDAR scan and learn trivial defense techniques.

# 1. Point Cloud Data Loading and Visualization

In [ ]:
# Install the Open3D library
# This step is needed because Open3D is not a standard library included in Google Colab
!pip install open3d

In [ ]:
# OPTIONAL: Run this code cell if you are using Google Colab
# Mount a Google Drive folder so that the data files can be accessed
from google.colab import drive
from google.colab import files
import sys
drive.mount('/content/drive', force_remount=True)
%cd drive/MyDrive/
sys.path.insert(0,'/content/drive/MyDrive/')

In [ ]:
# Import libraries and utility functions
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from utils import draw_geometries

In [ ]:
# Load point cloud from .txt file to a NumPy array
initial_pcd = None


print("Point cloud dimensions are: ", initial_pcd.shape)
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# Visualize the point cloud using a 3D web viewer
pcd_object = o3d.geometry.PointCloud()
pcd_object.points = o3d.utility.Vector3dVector(initial_pcd[:, 0:3])
draw_geometries([pcd_object], show_axes=True)

# 2. Point Cloud Filtering

In [ ]:
# Eliminate Ground Points

def filter_ground(cloud, ground_level=0):
    return cloud[cloud[:, 2] > ground_level, :]


# Filter out ground points
pcd_filtered = filter_ground(initial_pcd, -1.2)
print("Filtered point cloud from %d points to %d points" % (len(initial_pcd), len(pcd_filtered)))

# Visualize point cloud after filtering

pcd_object.points = o3d.utility.Vector3dVector(pcd_filtered[:, 0:3])
draw_geometries([pcd_object], show_axes=True)

In [ ]:
# Eliminate Distant Points

def filter_by_distance(cloud, distance=10):
    mask = np.sum(cloud[:,:2]**2, axis=1) < distance * distance
    #mask = np.sqrt(np.sum(cloud[:, :2]**2, axis=1)) < distance
    return cloud[mask, :]


pcd_filtered_2 = filter_by_distance(pcd_filtered, 10)
print("Filtered point cloud from %d points to %d points" % (len(pcd_filtered), len(pcd_filtered_2)))

pcd_object.points = o3d.utility.Vector3dVector(pcd_filtered_2)
draw_geometries([pcd_object], show_axes=True)

In [ ]:
# @title
# TODO: implementation function to perform Euclidean clustering at a specified threshold in meters
def euclidean_clustering(cloud, threshold=0.5):
    cluster_labels = np.zeros(len(cloud), dtype=int)
    cluster_idx = 1
    for i in range(len(cloud)):
        if cluster_labels[i] > 0:
            continue
        Q = [i]
        cluster_labels[i] = cluster_idx
        while len(Q) > 0:
            distances = np.sum((cloud - cloud[Q[-1]])**2, axis=1)
            neighbor_mask = distances < threshold * threshold
            neighbor_mask = np.logical_and(neighbor_mask, cluster_labels==0)
            cluster_labels[neighbor_mask] = cluster_idx
            del Q[-1]
            Q.extend(np.nonzero(neighbor_mask)[0])
        cluster_idx += 1
    return cluster_labels


In [ ]:
# Perform Clustering on Point Cloud

cluster_labels = euclidean_clustering(pcd_filtered_2)

print('Found %d clusters from %d points'%(cluster_labels.max(), len(pcd_filtered_2)))
pcd_object.points = o3d.utility.Vector3dVector(pcd_filtered_2)
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(cluster_labels)+1,3))
pcd_object.colors = o3d.utility.Vector3dVector(obj_color[cluster_labels])
draw_geometries([pcd_object])

# 3. Pointcloud Data Injection

In [ ]:
# TODO: Implement Injection of False Data (Spoofed Motor Bike and Human)


# Load in the the mesh "bike.ply" using the o3d.io.read_triangle_mesh() method and store in a variable named "mesh"

# Use the "sample_points_uniformly()" method of the mesh object to set the density of the mesh. Use the 'number_of_points' parameter
# and set it to 5000. Store the result in a variable named "bike".

# Represent the mesh as a numpy array and store it in a variable named "bike_pcd".

# Uncomment the next two lines of code to pre-process the pointcloud.
'''
bike_pcd = bike_pcd * 0.001
rotation_matrix_x = np.array([[1, 0, 0],[0, np.cos(np.deg2rad(90)), -np.sin(np.deg2rad(90))],[0, np.sin(np.deg2rad(90)), np.cos(np.deg2rad(90))]])
bike_pcd[:, 1] -= 6
bike_pcd = np.dot(bike_pcd, rotation_matrix_x.T)
bike_pcd[:, 2] += 4.3
'''


# Load in the the mesh "full_body.ply" using the o3d.io.read_triangle_mesh() method and store in a variable named "person_mesh"

# Use the "sample_points_uniformly()" method of the mesh object to set the density of the mesh. Use the 'number_of_points' parameter
# and set it to 2000. Store the result in a variable named "person".

# Represent the mesh as a numpy array and store it in a variable named "person_pcd".

# Uncomment the next five lines of code to pre-process the pointcloud.
'''
person_pcd = person_pcd * 0.001
person_pcd[:, 0] -= 2
rotation_matrix_x = np.array([[1, 0, 0],[0, np.cos(np.deg2rad(90)), -np.sin(np.deg2rad(90))],[0, np.sin(np.deg2rad(90)), np.cos(np.deg2rad(90))]])
person_pcd = np.dot(person_pcd, rotation_matrix_x.T)
person_pcd = person_pcd[person_pcd[:, 2] > -0.7,:]
'''

In [ ]:
# Use the np.vstack() function to create a union of the three pointcloud objects: pcd_filtered_2, bike_pcd, and person_pcd.
# Store the result in a variable named 'spoofed_pointcloud_data'.

# Perform clustering on the new spoofed pointcloud using the euclidean_clustering() function you created in Lab 2.
# Store the result in a variable named 'spoofed_labels'.

# Print out the shape of both the original pointcloud object and the new spoofed pointcloud object to see how they contrast.

In [ ]:
# Visualize the spoofed pointcloud
spoofed_pcd = o3d.geometry.PointCloud()
spoofed_pcd.points = o3d.utility.Vector3dVector(spoofed_pointcloud_data)
print('Found %d clusters from %d points'%(spoofed_labels.max(), len(spoofed_pointcloud_data)))
spoofed_labels = euclidean_clustering(spoofed_pointcloud_data)
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(spoofed_labels)+1,3))
spoofed_pcd.colors = o3d.utility.Vector3dVector(obj_color[spoofed_labels])
draw_geometries([spoofed_pcd])

## 4. Simulating a Laser Spoofer

In [ ]:
# TODO: Simulate a Laser Spoofer

# Laser Position
laser_pos = np.array([5.0, 5.0, 0.0])

# Implement the 'generate_spoof_laser_points()' function. This function will create spoofed laser points based on
# the position of the laser, number of points, and a spread radius.
def generate_spoof_laser_points(laser_pos, num_points=40, spread_radius=0.4):
    # x_vals and y_vals already completed
    x_vals = np.random.uniform(laser_pos[0] - spread_radius, laser_pos[0] + spread_radius/10, num_points)
    y_vals = np.random.uniform(laser_pos[1] - spread_radius, laser_pos[1] + spread_radius/10, num_points)

    # Use the np.random.uniform() function to create a distribution between laser_pos[2] - spread_radius*5
    # and laser_pos[2] + spread_radius*5. Store the result in a variable named z_vals.

    # Use the np.vstack() function to create a union of all three axis values. Store in a variable named spoofed_points.

    # Set spoofed_points equal to it's transpose (spoofed_points.T).

    # Return the new spoofed points.
    return None

In [ ]:
# Generate a set of spoofed laser points. Pass in laser_pos as an argument and store the result in a variable named
# spoofed_laser_points.

# Use the np.vstack() function to create a union of pcd_filtered_2 and spoofed_laser_points.
# Store the result in a variable named combined_points.

# Print out the shape of pcd_filtered_2 and combined_points

In [ ]:
# Visualize the spoofed laser points
spoofed_pcd = o3d.geometry.PointCloud()
spoofed_pcd.points = o3d.utility.Vector3dVector(combined_points)
draw_geometries([spoofed_pcd])

In [ ]:
# TODO: Perform clustering on combined_points with a threshold of 0.7.


In [ ]:
# Visualize with Clustering
print('Found %d clusters from %d points'%(combined_labels.max(), len(combined_points)))
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(combined_labels)+1,3))
spoofed_pcd.colors = o3d.utility.Vector3dVector(obj_color[combined_labels])
draw_geometries([spoofed_pcd])

## 5. Defense against Laser Spoofer

In [ ]:
# TODO: Implement functions for spoofing defense


# TODO: Implement the get_volume function. This function returns the volume of a passed in np array of 3D points, arr.
# Formula for Volume: L * W * H
def get_volume(arr):
  return None

# TODO: Implement the filter_spoofed_points() function.
def filter_spoofed_points(cloud, labels, threshold):
  new_cloud = None
  new_labels = None
  return new_cloud, new_labels


In [ ]:
# TODO: Run the new filter_spoofed_points() function and store the results in variables filtered_cloud and filtered_labels, respectively.
#       Pass in a threshold of 70 as an argument.

In [ ]:
# Visualize the filtered pointcloud data with no spoofed points
print('Found %d clusters from %d points'%(filtered_labels.max(), len(filtered_cloud)))
filtered_pcd = o3d.geometry.PointCloud()
filtered_pcd.points = o3d.utility.Vector3dVector(filtered_cloud)
color_sample_state = np.random.RandomState(0)
obj_color = color_sample_state.random((np.max(filtered_labels)+1,3))
filtered_pcd.colors = o3d.utility.Vector3dVector(obj_color[filtered_labels])
draw_geometries([filtered_pcd])